In [39]:
def relative_l2_error(model, exact_fn, n_x=200, n_t=100,
                      domain_x=(0, np.pi), domain_t=(0, 1)):
    x = np.linspace(domain_x[0], domain_x[1], n_x)
    t = np.linspace(domain_t[0], domain_t[1], n_t)
    X, T = np.meshgrid(x, t)
    inputs = np.stack([X.ravel(), T.ravel()], axis=1).astype(np.float32)
    p_pinn  = model(inputs).numpy().reshape(X.shape)
    p_exact = exact_fn(X, T)
    return np.linalg.norm(p_pinn - p_exact) / np.linalg.norm(p_exact)

## 1.1. Piezoconductivity Equation (Dirichlet)

Solve the 1D piezoconductivity equation: $p_t = \chi p_{xx}$ for $x$ in $[0, \pi]$, $t$ in $[0, 1]$, with initial condition $p(x, 0) = \sin(x)$, boundary conditions $p(0,t) = 0$, $p(\pi,t) = 0$.

In [40]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Parameters
chi = 1
domain_x = [0, np.pi]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    p = model(inputs)
    p_t = tape.gradient(p, t)
    p_x = tape.gradient(p, x)
  p_xx = tape.gradient(p_x, x)
  del tape
  return p_t - chi * p_xx

# Boundary condition loss (Dirichlet)
def bc_loss(model):
  t = tf.random.uniform(
    (50, 1),
    domain_t[0],
    domain_t[1]
  )
  x_left = tf.zeros_like(t)
  x_right = tf.ones_like(t) * np.pi
  inputs_left = tf.concat([x_left, t], axis=1)
  inputs_right = tf.concat([x_right, t], axis=1)
  p_left = model(inputs_left)
  p_right = model(inputs_right)

  return tf.reduce_mean(
      tf.square(p_left)
  ) + tf.reduce_mean(
      tf.square(p_right)
  )

# Initial condition loss
def ic_loss(model):
  x = tf.random.uniform(
      (50, 1),
      domain_x[0],
      domain_x[1]
  )
  t = tf.zeros_like(x)
  inputs = tf.concat([x, t], axis=1)
  p = model(inputs)
  p_ic = tf.sin(x)
  return tf.reduce_mean(tf.square(p-p_ic))

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
loss_history_dirichlet = []
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_bc = bc_loss(model)
    loss_ic = ic_loss(model)
    loss_dirichlet = loss_pde + loss_bc + loss_ic
  grads = tape.gradient(loss_dirichlet, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    loss_val = float(loss_dirichlet.numpy())
    loss_history_dirichlet.append(loss_val)
    print(f"Epoch {epoch}, Loss: {loss_dirichlet.numpy()}")

err_dirichlet = relative_l2_error(model, lambda x, t: np.exp(-t) * np.sin(x))
print(f"Dirichlet: relative L2 error = {err_dirichlet:.4e}")

Epoch 0, Loss: 0.6772328615188599
Epoch 50, Loss: 0.2475440502166748
Epoch 100, Loss: 0.18842117488384247
Epoch 150, Loss: 0.10509741306304932
Epoch 200, Loss: 0.0707293301820755
Epoch 250, Loss: 0.06110752373933792
Epoch 300, Loss: 0.04077157378196716
Epoch 350, Loss: 0.044796474277973175
Epoch 400, Loss: 0.031408339738845825
Epoch 450, Loss: 0.021601717919111252
Epoch 500, Loss: 0.02038971334695816
Epoch 550, Loss: 0.014207517728209496
Epoch 600, Loss: 0.011860903352499008
Epoch 650, Loss: 0.00874347984790802
Epoch 700, Loss: 0.010514708235859871
Epoch 750, Loss: 0.006501265335828066
Epoch 800, Loss: 0.006478527560830116
Epoch 850, Loss: 0.005344124510884285
Epoch 900, Loss: 0.003840475343167782
Epoch 950, Loss: 0.004025286063551903
Dirichlet: relative L2 error = 3.7236e-02


In [41]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  p_plot = model(inputs_plot).numpy()
  p_exact = np.exp(-t_val) * np.sin(x_plot)
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': p_plot.flatten(),
      'Exact': p_exact.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')
fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='p(x, t)'
)
fig.show()

## 1.2. Piezoconductivity Equation (Neumann)

Solve the 1D piezoconductivity equation: $p_t = \chi p_{xx}$ for $x$ in $[0, \pi]$, $t$ in $[0, 1]$, with initial condition $p(x, 0) = \cos(x)$, boundary conditions $p_x(0,t) = 0$, $p_x(\pi,t) = 0$.

In [42]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Parameters
chi = 1
domain_x = [0, np.pi]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    p = model(inputs)
    p_t = tape.gradient(p, t)
    p_x = tape.gradient(p, x)
  p_xx = tape.gradient(p_x, x)
  del tape
  return p_t - chi * p_xx

# Boundary condition loss (Neumann)
def bc_loss(model):
  t = tf.random.uniform(
    (50, 1),
    domain_t[0],
    domain_t[1]
  )
  x_left = tf.zeros_like(t)
  x_right = tf.ones_like(t) * np.pi
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x_left)
    tape.watch(x_right)
    inputs_left = tf.concat([x_left, t], axis=1)
    inputs_right = tf.concat([x_right, t], axis=1)
    p_left = model(inputs_left)
    p_right = model(inputs_right)

    p_x_left = tape.gradient(p_left, x_left)
    p_x_right = tape.gradient(p_right, x_right)
  del tape

  return tf.reduce_mean(
      tf.square(p_x_left)
  ) + tf.reduce_mean(
      tf.square(p_x_right)
  )

# Initial condition loss
def ic_loss(model):
  x = tf.random.uniform(
      (50, 1),
      domain_x[0],
      domain_x[1]
  )
  t = tf.zeros_like(x)
  inputs = tf.concat([x, t], axis=1)
  p = model(inputs)
  p_ic = tf.cos(x)
  return tf.reduce_mean(tf.square(p-p_ic))

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
loss_history_neumann = []
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_bc = bc_loss(model)
    loss_ic = ic_loss(model)
    loss_neumann = loss_pde + loss_bc + loss_ic
  grads = tape.gradient(loss_neumann, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    loss_val = float(loss_neumann.numpy())
    loss_history_neumann.append(loss_val)
    print(f"Epoch {epoch}, Loss: {loss_neumann.numpy()}")

err_neumann = relative_l2_error(model, lambda x, t: np.exp(-t) * np.cos(x))
print(f"Neumann: relative L2 error = {err_neumann:.4e}")

Epoch 0, Loss: 1.3950021266937256
Epoch 50, Loss: 0.3394886255264282
Epoch 100, Loss: 0.17860189080238342
Epoch 150, Loss: 0.1628263145685196
Epoch 200, Loss: 0.11140841245651245
Epoch 250, Loss: 0.09943348914384842
Epoch 300, Loss: 0.07251469045877457
Epoch 350, Loss: 0.06018407642841339
Epoch 400, Loss: 0.0391002781689167
Epoch 450, Loss: 0.03866967558860779
Epoch 500, Loss: 0.025710105895996094
Epoch 550, Loss: 0.015373422764241695
Epoch 600, Loss: 0.012152813374996185
Epoch 650, Loss: 0.00968115869909525
Epoch 700, Loss: 0.006338147446513176
Epoch 750, Loss: 0.007558577228337526
Epoch 800, Loss: 0.005592076573520899
Epoch 850, Loss: 0.0051445988938212395
Epoch 900, Loss: 0.004417107906192541
Epoch 950, Loss: 0.0050486172549426556
Neumann: relative L2 error = 3.1281e-02


In [43]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  p_plot = model(inputs_plot).numpy()
  p_exact = np.exp(-t_val) * np.cos(x_plot)
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': p_plot.flatten(),
      'Exact': p_exact.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')
fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='p(x, t)'
)
fig.show()

## 1.3. Piezoconductivity Equation (Robin)

Solve the 1D piezoconductivity equation: $p_t = \chi p_{xx}$ for $x$ in $[0, \pi]$, $t$ in $[0, 1]$, with initial condition $p(x, 0) = \cos\left(x - \frac{\pi}{4}\right)$, boundary conditions $p_x(0,t) - p(0,t) = 0$, $p_x(\pi,t) - p(\pi,t) = 0$.

In [44]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Parameters
chi = 1
domain_x = [0, np.pi]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    p = model(inputs)
    p_t = tape.gradient(p, t)
    p_x = tape.gradient(p, x)
  p_xx = tape.gradient(p_x, x)
  del tape
  return p_t - chi * p_xx

# Boundary condition loss (Robin)
def bc_loss(model):
  t = tf.random.uniform(
    (50, 1),
    domain_t[0],
    domain_t[1]
  )
  x_left = tf.zeros_like(t)
  x_right = tf.ones_like(t) * np.pi
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x_left)
    tape.watch(x_right)
    inputs_left = tf.concat([x_left, t], axis=1)
    inputs_right = tf.concat([x_right, t], axis=1)
    p_left = model(inputs_left)
    p_right = model(inputs_right)

    p_x_left = tape.gradient(p_left, x_left)
    p_x_right = tape.gradient(p_right, x_right)
  del tape

  loss_left = tf.reduce_mean(tf.square(p_x_left - p_left))
  loss_right = tf.reduce_mean(tf.square(p_x_right - p_right))

  return loss_left + loss_right

# Initial condition loss
def ic_loss(model):
  x = tf.random.uniform(
      (50, 1),
      domain_x[0],
      domain_x[1]
  )
  t = tf.zeros_like(x)
  inputs = tf.concat([x, t], axis=1)
  p = model(inputs)
  p_ic = tf.cos(x - np.pi / 4)
  return tf.reduce_mean(tf.square(p-p_ic))

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
loss_history_robin = []
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_bc = bc_loss(model)
    loss_ic = ic_loss(model)
    loss_robin = loss_pde + loss_bc + loss_ic
  grads = tape.gradient(loss_robin, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    loss_val = float(loss_robin.numpy())
    loss_history_robin.append(loss_val)
    print(f"Epoch {epoch}, Loss: {loss_robin.numpy()}")

err_robin = relative_l2_error(model, lambda x, t: np.exp(-t) * np.cos(x - np.pi / 4))
print(f"Robin: relative L2 error = {err_robin:.4e}")

Epoch 0, Loss: 1.0925776958465576
Epoch 50, Loss: 0.3017493486404419
Epoch 100, Loss: 0.23753978312015533
Epoch 150, Loss: 0.1518479883670807
Epoch 200, Loss: 0.09297291934490204
Epoch 250, Loss: 0.05078268051147461
Epoch 300, Loss: 0.030827825888991356
Epoch 350, Loss: 0.020785940811038017
Epoch 400, Loss: 0.01864568144083023
Epoch 450, Loss: 0.014809718355536461
Epoch 500, Loss: 0.01035866979509592
Epoch 550, Loss: 0.012091703712940216
Epoch 600, Loss: 0.008134362287819386
Epoch 650, Loss: 0.007270229049026966
Epoch 700, Loss: 0.0069198706187307835
Epoch 750, Loss: 0.0055372463539242744
Epoch 800, Loss: 0.006942734122276306
Epoch 850, Loss: 0.006551862694323063
Epoch 900, Loss: 0.00591575913131237
Epoch 950, Loss: 0.0050297086127102375
Robin: relative L2 error = 1.9449e-01


In [45]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  p_plot = model(inputs_plot).numpy()
  p_exact = np.exp(-t_val) * np.cos(x_plot - np.pi / 4)
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': p_plot.flatten(),
      'Exact': p_exact.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')
fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='p(x, t)'
)
fig.show()

## LOSS:

In [46]:
import plotly.graph_objects as go

epochs = [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500,
          550, 600, 650, 700, 750, 800, 850, 900, 950]

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=loss_history_dirichlet, mode='lines+markers', name='Дирихле'))
fig.add_trace(go.Scatter(x=epochs, y=loss_history_neumann, mode='lines+markers', name='Нейман'))
fig.add_trace(go.Scatter(x=epochs, y=loss_history_robin, mode='lines+markers', name='Робин'))
fig.update_layout(
    title='Сходимость обучения PINN',
    xaxis_title='Эпоха',
    yaxis_title='Функция потерь',
    yaxis_type='log',  # логарифмическая шкала
    template='plotly_white'
)
fig.show()